# ISIC 2018 Task 3: Skin Lesion Classification with EfficientNet-B0

7-class skin lesion classification fine-tuned from ImageNet-pretrained EfficientNet-B0.

**Dataset:** HAM10000 — Human Against Machine with 10000 training images  
**Reference:** Tschandl et al., Scientific Data, 2018. https://doi.org/10.1038/sdata.2018.161  
**Challenge:** Codella et al., ISIC 2018, arXiv:1902.03368, 2019  
**Kaggle dataset:** `kmader/skin-cancer-mnist-ham10000`

**Primary metric:** Balanced accuracy (mean per-class recall), per Codella et al. (2019).  
Standard accuracy is not used because NV accounts for 67% of training images.

**Class imbalance handling:** Weighted cross-entropy with per-class weights  
`weight_c = total_samples / (num_classes * count_c)`.

## 1. Imports and configuration

In [ ]:
import random
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.utils.data
import torchvision.transforms as T
import timm
from PIL import Image
from sklearn.metrics import balanced_accuracy_score
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt

In [ ]:
# ---------------------------------------------------------------------------
# Configuration
# ---------------------------------------------------------------------------
BASE = Path("/kaggle/input/skin-cancer-mnist-ham10000")

# HAM10000 images are split across two part directories.
IMG_DIRS = [
    BASE / "HAM10000_images_part_1",
    BASE / "HAM10000_images_part_2",
]

CFG = {
    "img_dirs":    IMG_DIRS,
    "metadata":    BASE / "HAM10000_metadata.csv",
    "ckpt_dir":    Path("/kaggle/working/checkpoints"),

    # Data
    "val_fraction": 0.2,
    "img_size":     224,       # EfficientNet-B0 canonical input size
    "seed":         42,

    # Training
    "num_classes":  7,
    "batch_size":   32,
    "num_epochs":   20,
    "lr":           1e-4,
    "weight_decay": 1e-4,
    "lr_step_size": 7,
    "lr_gamma":     0.5,
}

CFG["ckpt_dir"].mkdir(parents=True, exist_ok=True)

# Reproducibility
random.seed(CFG["seed"])
np.random.seed(CFG["seed"])
torch.manual_seed(CFG["seed"])
torch.cuda.manual_seed_all(CFG["seed"])

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")

## 2. Dataset

In [ ]:
# Class label mapping (alphabetical for determinism)
CLASSES = ["akiec", "bcc", "bkl", "df", "mel", "nv", "vasc"]
CLASS_TO_IDX = {c: i for i, c in enumerate(CLASSES)}


def build_image_index(img_dirs: list[Path]) -> dict[str, Path]:
    """
    Build a mapping from image_id to full file path by scanning all image
    directories. HAM10000 splits images across two part directories.

    Args:
        img_dirs: List of directories containing ISIC_XXXXXXX.jpg files.

    Returns:
        Dict mapping image_id (e.g. 'ISIC_0024306') to its Path.
    """
    index = {}
    for d in img_dirs:
        for p in d.glob("*.jpg"):
            index[p.stem] = p
    return index


class ISICClassificationDataset(torch.utils.data.Dataset):
    """
    HAM10000 skin lesion classification dataset.

    Returns (image_tensor, label) pairs where label is an integer in [0, 6]
    corresponding to one of seven diagnostic classes.

    Args:
        records:   DataFrame with columns ['image_id', 'dx'] for this split.
        img_index: Dict mapping image_id to file path (from build_image_index).
        transform: torchvision transform applied to each PIL image.
    """

    def __init__(
        self,
        records: pd.DataFrame,
        img_index: dict[str, Path],
        transform: T.Compose,
    ) -> None:
        self.records = records.reset_index(drop=True)
        self.img_index = img_index
        self.transform = transform

    def __len__(self) -> int:
        return len(self.records)

    def __getitem__(self, idx: int) -> tuple[torch.Tensor, int]:
        row = self.records.iloc[idx]
        image = Image.open(self.img_index[row["image_id"]]).convert("RGB")
        return self.transform(image), CLASS_TO_IDX[row["dx"]]

In [ ]:
def build_dataloaders(cfg: dict) -> tuple[
    torch.utils.data.DataLoader,
    torch.utils.data.DataLoader,
    torch.Tensor,
]:
    """
    Build train and validation DataLoaders with a lesion-level stratified split.

    The split is performed on lesion_id rather than image_id because the same
    lesion appears multiple times in HAM10000 with different crops. Splitting
    on image_id would leak lesion information into validation and inflate metrics.

    Returns:
        Tuple of (train_loader, val_loader, class_weights) where class_weights
        is a FloatTensor of shape (num_classes,) for use with CrossEntropyLoss.
    """
    meta = pd.read_csv(cfg["metadata"])
    img_index = build_image_index(cfg["img_dirs"])

    # Stratified split on unique lesion IDs to prevent data leakage.
    # Each lesion_id maps to exactly one dx class.
    lesion_df = meta.drop_duplicates("lesion_id")[["lesion_id", "dx"]]
    train_lesions, val_lesions = train_test_split(
        lesion_df["lesion_id"].values,
        test_size=cfg["val_fraction"],
        stratify=lesion_df["dx"].values,
        random_state=cfg["seed"],
    )
    train_lesions = set(train_lesions)
    val_lesions   = set(val_lesions)

    train_records = meta[meta["lesion_id"].isin(train_lesions)]
    val_records   = meta[meta["lesion_id"].isin(val_lesions)]

    # Per-class weights from training distribution: weight_c = total / (C * count_c)
    class_counts = train_records["dx"].value_counts()
    total = len(train_records)
    weights = torch.tensor(
        [total / (cfg["num_classes"] * class_counts[c]) for c in CLASSES],
        dtype=torch.float32,
    )

    # ImageNet normalisation statistics match timm EfficientNet-B0 pretrained weights.
    mean = (0.485, 0.456, 0.406)
    std  = (0.229, 0.224, 0.225)

    train_transform = T.Compose([
        T.Resize((cfg["img_size"], cfg["img_size"])),
        T.RandomHorizontalFlip(),
        T.RandomVerticalFlip(),
        T.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.1),
        T.ToTensor(),
        T.Normalize(mean=mean, std=std),
    ])
    val_transform = T.Compose([
        T.Resize((cfg["img_size"], cfg["img_size"])),
        T.ToTensor(),
        T.Normalize(mean=mean, std=std),
    ])

    train_ds = ISICClassificationDataset(train_records, img_index, train_transform)
    val_ds   = ISICClassificationDataset(val_records,   img_index, val_transform)

    train_loader = torch.utils.data.DataLoader(
        train_ds, batch_size=cfg["batch_size"], shuffle=True,
        num_workers=2, pin_memory=True, drop_last=True,
    )
    val_loader = torch.utils.data.DataLoader(
        val_ds, batch_size=cfg["batch_size"], shuffle=False,
        num_workers=2, pin_memory=True,
    )

    print(f"Train: {len(train_ds)} images | Val: {len(val_ds)} images")
    print("Class weights:")
    for c, w in zip(CLASSES, weights):
        print(f"  {c:6s}: {w:.3f}")

    return train_loader, val_loader, weights


train_loader, val_loader, class_weights = build_dataloaders(CFG)
class_weights = class_weights.to(DEVICE)

## 3. Dataset sanity check

In [ ]:
def visualise_samples(loader: torch.utils.data.DataLoader, n: int = 8) -> None:
    """Plot n training images with their class labels."""
    images, labels = next(iter(loader))
    n = min(n, len(images))
    mean = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
    std  = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)

    fig, axes = plt.subplots(1, n, figsize=(3 * n, 3))
    for ax, img, lbl in zip(axes, images[:n], labels[:n]):
        img = (img * std + mean).clamp(0, 1).permute(1, 2, 0).numpy()
        ax.imshow(img)
        ax.set_title(CLASSES[lbl.item()], fontsize=8)
        ax.axis("off")
    plt.tight_layout()
    plt.show()


visualise_samples(train_loader, n=8)

## 4. Model

In [ ]:
def build_model(num_classes: int) -> nn.Module:
    """
    Load EfficientNet-B0 pretrained on ImageNet and replace the classification
    head for num_classes output classes.

    Passing num_classes directly to timm.create_model replaces the head in one
    call, avoiding manual surgery on the classifier layer.

    Args:
        num_classes: Number of output classes.

    Returns:
        EfficientNet-B0 model ready for fine-tuning.
    """
    return timm.create_model(
        "efficientnet_b0",
        pretrained=True,
        num_classes=num_classes,
    )


model = build_model(CFG["num_classes"]).to(DEVICE)
n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Trainable parameters: {n_params:,}")

## 5. Training and evaluation

In [ ]:
def train_one_epoch(
    model: nn.Module,
    loader: torch.utils.data.DataLoader,
    optimizer: torch.optim.Optimizer,
    criterion: nn.Module,
) -> float:
    """
    Run one training epoch.

    Args:
        model:     EfficientNet-B0 model in train mode.
        loader:    Training DataLoader.
        optimizer: Optimizer.
        criterion: Weighted CrossEntropyLoss.

    Returns:
        Mean loss over the epoch.
    """
    model.train()
    total_loss = 0.0

    for images, labels in loader:
        images = images.to(DEVICE)
        labels = labels.to(DEVICE)

        loss = criterion(model(images), labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    return total_loss / len(loader)


@torch.no_grad()
def evaluate(
    model: nn.Module,
    loader: torch.utils.data.DataLoader,
    criterion: nn.Module,
) -> tuple[float, float, float]:
    """
    Evaluate the model and return loss, accuracy, and balanced accuracy.

    All predictions and labels are accumulated across the full validation set
    before computing metrics. Computing balanced accuracy per batch and averaging
    would be incorrect because batches have different class distributions.

    Args:
        model:     EfficientNet-B0 model in eval mode.
        loader:    Validation DataLoader.
        criterion: Loss function (same weighted CrossEntropyLoss as training).

    Returns:
        Tuple of (mean_loss, accuracy, balanced_accuracy).
    """
    model.eval()
    all_preds:  list[np.ndarray] = []
    all_labels: list[np.ndarray] = []
    total_loss = 0.0

    for images, labels in loader:
        images = images.to(DEVICE)
        labels = labels.to(DEVICE)

        logits = model(images)
        total_loss += criterion(logits, labels).item()

        all_preds.append(logits.argmax(dim=1).cpu().numpy())
        all_labels.append(labels.cpu().numpy())

    all_preds  = np.concatenate(all_preds)
    all_labels = np.concatenate(all_labels)

    return (
        total_loss / len(loader),
        float((all_preds == all_labels).mean()),
        float(balanced_accuracy_score(all_labels, all_preds)),
    )


def train(
    model: nn.Module,
    train_loader: torch.utils.data.DataLoader,
    val_loader: torch.utils.data.DataLoader,
    cfg: dict,
) -> tuple[dict, int]:
    """
    Full training loop with LR scheduling and best-checkpoint saving.

    Best checkpoint is saved by validation balanced accuracy, which is the
    primary metric per Codella et al. (2019). If cfg['resume_ckpt'] points
    to an existing file, resumes from that checkpoint and runs cfg['num_epochs']
    additional epochs. Otherwise trains from scratch.

    Args:
        model:        EfficientNet-B0 model.
        train_loader: Training DataLoader.
        val_loader:   Validation DataLoader.
        cfg:          Configuration dict.

    Returns:
        Tuple of (history dict, start_epoch) for use in plotting.
    """
    criterion = nn.CrossEntropyLoss(weight=class_weights)

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=cfg["lr"],
        weight_decay=cfg["weight_decay"],
    )
    scheduler = torch.optim.lr_scheduler.StepLR(
        optimizer, step_size=cfg["lr_step_size"], gamma=cfg["lr_gamma"]
    )

    history = {"train_loss": [], "val_loss": [], "val_acc": [], "val_bal_acc": []}
    best_bal_acc, start_epoch = 0.0, 1

    if cfg.get("resume_ckpt") and Path(cfg["resume_ckpt"]).exists():
        ckpt = torch.load(cfg["resume_ckpt"], map_location=DEVICE)
        model.load_state_dict(ckpt["model_state_dict"])
        optimizer.load_state_dict(ckpt["optimizer_state_dict"])
        best_bal_acc = ckpt["val_bal_acc"]
        start_epoch  = ckpt["epoch"] + 1
        for _ in range(ckpt["epoch"]):  # fast-forward scheduler to correct LR
            scheduler.step()
        print(f"Resumed from epoch {ckpt['epoch']} (val_bal_acc={best_bal_acc:.4f})")

    for epoch in range(start_epoch, start_epoch + cfg["num_epochs"]):
        train_loss = train_one_epoch(model, train_loader, optimizer, criterion)
        scheduler.step()
        val_loss, val_acc, val_bal_acc = evaluate(model, val_loader, criterion)

        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_loss)
        history["val_acc"].append(val_acc)
        history["val_bal_acc"].append(val_bal_acc)

        print(
            f"Epoch {epoch:02d}/{start_epoch + cfg['num_epochs'] - 1} | "
            f"train_loss={train_loss:.4f} | "
            f"val_loss={val_loss:.4f} | "
            f"val_acc={val_acc:.4f} | "
            f"val_bal_acc={val_bal_acc:.4f}"
        )

        if val_bal_acc > best_bal_acc:
            best_bal_acc = val_bal_acc
            torch.save(
                {
                    "epoch": epoch,
                    "model_state_dict": model.state_dict(),
                    "optimizer_state_dict": optimizer.state_dict(),
                    "val_bal_acc": best_bal_acc,
                },
                cfg["ckpt_dir"] / "best_efficientnet_b0_ham10000.pth",
            )
            print(f"  Checkpoint saved (val_bal_acc={best_bal_acc:.4f})")

    print(f"\nTraining complete. Best val balanced accuracy: {best_bal_acc:.4f}")
    return history, start_epoch


CFG["num_epochs"]  = 20
CFG["resume_ckpt"] = CFG["ckpt_dir"] / "best_efficientnet_b0_ham10000.pth"

history, start_epoch = train(model, train_loader, val_loader, CFG)

## 6. Training curves

In [ ]:
epochs = range(start_epoch, start_epoch + len(history["train_loss"]))

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].plot(epochs, history["train_loss"], marker="o", label="Train loss")
axes[0].plot(epochs, history["val_loss"],   marker="o", label="Val loss")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Loss")
axes[0].set_title("Loss")
axes[0].legend()
axes[0].grid(True)

axes[1].plot(epochs, history["val_acc"], marker="o", color="steelblue")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Accuracy")
axes[1].set_title("Validation Accuracy")
axes[1].grid(True)

axes[2].plot(epochs, history["val_bal_acc"], marker="o", color="darkorange",
             label="Val balanced accuracy")
axes[2].set_xlabel("Epoch")
axes[2].set_ylabel("Balanced Accuracy")
axes[2].set_title("Validation Balanced Accuracy (primary metric)")
axes[2].legend()
axes[2].grid(True)

plt.tight_layout()
plt.savefig("/kaggle/working/training_curves_task3.png", dpi=150)
plt.show()

## 7. Per-class evaluation on validation set

In [ ]:
# Load best checkpoint
ckpt = torch.load(
    CFG["ckpt_dir"] / "best_efficientnet_b0_ham10000.pth", map_location=DEVICE
)
model.load_state_dict(ckpt["model_state_dict"])
print(f"Loaded checkpoint from epoch {ckpt['epoch']} "
      f"(val_bal_acc={ckpt['val_bal_acc']:.4f})")


@torch.no_grad()
def per_class_report(
    model: nn.Module,
    loader: torch.utils.data.DataLoader,
) -> None:
    """Print per-class recall (the component terms of balanced accuracy)."""
    model.eval()
    all_preds:  list[np.ndarray] = []
    all_labels: list[np.ndarray] = []

    for images, labels in loader:
        logits = model(images.to(DEVICE))
        all_preds.append(logits.argmax(dim=1).cpu().numpy())
        all_labels.append(labels.numpy())

    all_preds  = np.concatenate(all_preds)
    all_labels = np.concatenate(all_labels)

    print(f"{'Class':8s} {'Recall':>8s} {'N val':>8s}")
    print("-" * 28)
    for i, cls in enumerate(CLASSES):
        mask   = all_labels == i
        recall = float((all_preds[mask] == i).mean()) if mask.any() else 0.0
        print(f"{cls:8s} {recall:8.4f} {mask.sum():8d}")
    print("-" * 28)
    print(f"{'Bal acc':8s} {balanced_accuracy_score(all_labels, all_preds):8.4f}")


per_class_report(model, val_loader)

## 8. Final validation score

In [ ]:
criterion = nn.CrossEntropyLoss(weight=class_weights)
_, final_acc, final_bal_acc = evaluate(model, val_loader, criterion)
print(f"Final val accuracy:          {final_acc:.4f}")
print(f"Final val balanced accuracy: {final_bal_acc:.4f}")
print(f"Best val balanced accuracy:  {ckpt['val_bal_acc']:.4f}")